# Week 0 — Segment 5 Workshop: What Is Training?

**Companion notebook to `README.md` Segment 5.** Open this in Google Colab or any Jupyter environment with a small GPU.

**Goal:** do an actual end-to-end fine-tune of a small LLM in ~15 minutes of runtime, and see what training changes and what it doesn't.

**Recommended environment:** Google Colab, free T4 GPU tier.
- `Runtime → Change runtime type → T4 GPU → Save`
- Total runtime: ~15-20 min on T4.

**Alternative environments:**
- Apple Silicon Mac with `mlx-lm` — roughly 2x slower, fully local. You'll need to port the training step.
- Any CUDA GPU with 8GB+ VRAM works.
- CPU only: you can still run inference (skip Step 4), but the before/after demo won't have an 'after.'

Work through the cells top to bottom. Read the markdown headers — they narrate what's happening and why.

## Step 1 — Install and load a small instruction-tuned base model

We use **Qwen 2.5 0.5B Instruct** — a real instruction-tuned model from Alibaba. Small enough to fine-tune in a few minutes on a free Colab GPU, big enough to show real before/after behavior change.

The `pip install` line installs the Hugging Face stack:
- `transformers` — the model-loading library
- `peft` — parameter-efficient fine-tuning (what gives us LoRA)
- `trl` — the trainer loop for SFT
- `datasets` — dataset utilities
- `accelerate` + `bitsandbytes` — GPU plumbing

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters() / 1e6:.0f}M")

## Step 2 — Inference BEFORE fine-tuning

Run the model on three support-ticket-style prompts. These are our baseline. **Save the outputs mentally (or copy them into a cell below)** — we'll compare against them after training.

Expected observation: the answers are fine but probably *inconsistent in shape*. You might get 'Authentication,' 'account access,' or 'The category is: billing.' The model has no idea we want one-word answers from a fixed vocabulary.

In [ ]:
def chat(prompt, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

prompts = [
    "Customer says: 'my account is locked after too many login attempts.' Categorize this ticket in one word.",
    "Customer says: 'the checkout button is greyed out on the mobile app.' Categorize this ticket in one word.",
    "Customer says: 'I was charged twice for the same subscription.' Categorize this ticket in one word.",
]

print("=== BEFORE fine-tuning ===\n")
before_outputs = []
for p in prompts:
    ans = chat(p)
    before_outputs.append(ans)
    print(f"Q: {p}")
    print(f"A: {ans}")
    print("---")

## Step 3 — Build a tiny instruction dataset

In a real fine-tune you'd have thousands of examples. For the workshop, ~30 is enough to see the shape change.

Each example is a `(user, assistant)` pair. The assistant answer is **always one word from a fixed vocabulary**: `auth`, `bug`, `billing`, or `howto`. This is the *shape* we're teaching the model to produce.

Feel free to edit the list below — add more examples, change the categories, see what happens.

In [ ]:
from datasets import Dataset

TRAINING_EXAMPLES = [
    # auth — login / password / 2FA issues
    {"user": "Customer says: 'I can't log in, keeps saying wrong password.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'forgot my password and reset link isn't coming.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'two-factor code never arrived.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'my SSO login just redirects me back to the login page.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'I got locked out after entering the wrong password three times.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'my session keeps expiring every five minutes.' Categorize this ticket in one word.", "assistant": "auth"},
    {"user": "Customer says: 'the magic link in my email doesn't work.' Categorize this ticket in one word.", "assistant": "auth"},
    # bug — things that are broken / crashing / not working
    {"user": "Customer says: 'the dashboard is stuck loading.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'app crashes when I click settings.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'report export produces empty file.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'uploading a CSV fails with a 500 error.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'charts render upside down on Safari.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'notifications haven't been sending since yesterday.' Categorize this ticket in one word.", "assistant": "bug"},
    {"user": "Customer says: 'the search bar returns no results for anything.' Categorize this ticket in one word.", "assistant": "bug"},
    # billing — money / invoices / charges
    {"user": "Customer says: 'I was charged twice for last month.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'invoice doesn't match what I signed up for.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'please cancel my subscription and refund me.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'my card was declined but the plan still activated.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'I was promised a 20% discount that never applied.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'I need a receipt for my last three payments.' Categorize this ticket in one word.", "assistant": "billing"},
    {"user": "Customer says: 'can I switch from monthly to annual and get a prorated credit?' Categorize this ticket in one word.", "assistant": "billing"},
    # howto — questions about how to do / find / configure something
    {"user": "Customer says: 'how do I invite a new team member?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'where do I find the API keys?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'what's the difference between the Pro and Team plans?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'how can I export my data to JSON?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'can you walk me through setting up SSO?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'what's the best way to share a dashboard with a customer?' Categorize this ticket in one word.", "assistant": "howto"},
    {"user": "Customer says: 'how do I change the owner of my organization?' Categorize this ticket in one word.", "assistant": "howto"},
]

print(f"Training examples: {len(TRAINING_EXAMPLES)}")

def format_example(ex):
    messages = [
        {"role": "user", "content": ex["user"]},
        {"role": "assistant", "content": ex["assistant"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(TRAINING_EXAMPLES).map(format_example)
print("\nSample formatted training example:\n")
print(dataset[0]["text"])

## Step 4 — Run a LoRA fine-tune

**LoRA (Low-Rank Adaptation)** is the cheap kind of fine-tuning. Instead of updating all ~500M weights in the model, we add a tiny set of 'adapter' weights (typically 1-5% of the total) and only train those. For most purposes this captures the same behavior change for a tiny fraction of the cost — no need to store or ship a full model copy.

**What to watch:** the `loss` value in the logs should decrease as training progresses. That's the model getting better at producing the shape we're teaching it.

**Expected runtime:** 5-10 minutes on a T4. If you're on CPU, skip this cell — you'll only see the 'before' side of the demo.

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(
    output_dir="./lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

## Step 5 — Inference AFTER fine-tuning

Run the **exact same three prompts** from Step 2 and compare.

**Expected observation:** answers are now short, one-word, from the trained vocabulary — `auth`, `bug`, `billing`, `howto`. Consistent shape. **This is the fine-tune working.**

In [ ]:
print("=== AFTER fine-tuning ===\n")
after_outputs = []
for p in prompts:
    ans = chat(p, max_new_tokens=20)
    after_outputs.append(ans)
    print(f"Q: {p}")
    print(f"A: {ans}")
    print("---")

print("\n=== Side by side ===\n")
for p, b, a in zip(prompts, before_outputs, after_outputs):
    print(f"Q: {p[:70]}...")
    print(f"  BEFORE: {b.strip()[:80]}")
    print(f"  AFTER:  {a.strip()[:80]}")
    print()

## Step 6 — Probe the limits

Now the interesting part: try prompts that are **outside** the training distribution. What does the fine-tuned model do when asked something it wasn't trained on?

In [ ]:
extra_prompts = [
    # Looks like support but isn't a real category we trained
    "Customer says: 'the pricing page has a typo.' Categorize this ticket in one word.",
    # Positive feedback — no 'feedback' category in training
    "Customer says: 'I love your product!' Categorize this ticket in one word.",
    # Completely off-topic
    "What is the capital of France?",
]

for p in extra_prompts:
    ans = chat(p, max_new_tokens=30)
    print(f"Q: {p}")
    print(f"A: {ans}")
    print("---")

## What you just learned

Four takeaways worth keeping:

1. **Fine-tuning teaches a shape, not facts.** The model didn't learn what your product is. It learned a consistent response pattern from the examples you showed it. The capital of France didn't change.
2. **The model generalizes within the shape you showed it.** Prompts that look like training data get handled well. Prompts outside it get forced into the wrong shape — the 'typo' ticket might have landed in `bug` or `howto`; the compliment probably didn't get a category it deserved.
3. **LoRA is cheap.** The trainable-parameter count you saw in Step 4 was 1-5% of the model. That sliver of parameters captured almost all of the behavior change. Full fine-tuning is rarely necessary.
4. **Data is the lever, not compute.** The 30-example change was driven by the *examples*, not the training algorithm. Better fine-tunes almost always come from better data, not more epochs.

## Your deliverable (from README Segment 5)

Write one paragraph: **what changed** in the model after training, **what didn't change,** and **one problem you'd NOT solve with fine-tuning.**

## Decision framework — when to fine-tune?

| Problem | Try this first | Fine-tune? |
|---|---|---|
| Model doesn't know our facts | RAG (retrieval) | Rarely |
| Tone doesn't match our brand | System prompt + few-shot | Maybe |
| Output format is wrong | Structured outputs / JSON mode | Rarely |
| Too expensive at scale | Smaller model, or distill | Yes, via distillation |
| Model refuses reasonable requests | Different base model or provider | Sometimes |
| Need a specialized tiny model | Yes | **Yes, this is the real use case** |

**Heuristic:** if a good system prompt gets you 80% of the way, fine-tuning is rarely worth the operational cost.

## Where to go next

- If this workshop lit you up and you want to go deep on how models are built: **[How Tuning Works](../How-Tuning-Works/README.md)** — Week 2 specifically is a full SFT treatment.
- If you want to evaluate and monitor AI in production without necessarily building it: **[Data Science for Fusion Engineers](../DataScienceForFusionEngineers.md)** — Week 08-Alt covers modern eval methods.
- If this was enough: you're set. Go back to `README.md` Segment 6 for the honest decision rubric.